In [1]:
import glob

import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast
from utils.hist_utils import get_highest_channels, fit_and_scale_mc

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.Declare('#include "const.h"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

In [2]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/hal/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)

data_files = []

for i in range(1, 46, 1):
    data_files += glob.glob(
        f"/workspace/root_files/hal/DATA_SEMILEPTONIC_NoSmearing/dk0*_{i}.root"
    )

# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

# Scaling factor
lumi_data = 817216.9875
lumi_mc = 745800.1127
scaling_factor = 45 / 135
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

mPi0 = 134.9768 # MeV
mPiCh = 139.57039 # MeV
mK0 = 497.611 # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################    
    .Filter("mcflag == 0 && mctruth != -1")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################    
    .Filter("mcflag == 1 && mctruth != -1")
)

# Filtrowanie dla próbki semileptonicznej (dziedziczy wszystkie nowe kolumny m_*_hyp)
df_Semileptonic = df.Filter(
    "mcflag == 1 && (mctruth == 5 || mctruth == 0 || mctruth == -1)"
)

# df Signal
df_Signal = df.Filter(
    f"mcflag == 1 && mctruth == 1"
)

Przetestowano 135 plików. Poprawnych: 135, Odrzuconych: 0
Przetestowano 45 plików. Poprawnych: 45, Odrzuconych: 0
Lumi factor: 0.3333


In [3]:
stats_semileptonic = {"counts": [], "efficiency": []}
counts_error_code = {300: 0, 301: 0, 303: 0, 307: 0}

efficiency_error_code = {code: 0 for code in counts_error_code.keys()}
relative_efficiency_error_code = {code: 0 for code in counts_error_code.keys()}

# General counts
stats_semileptonic["counts"].append(df_Semileptonic.Count())
stats_semileptonic["counts"].append(df_Semileptonic.Filter("mctruth == 0 || mctruth == 5").Count())
stats_semileptonic["counts"].append(df_Semileptonic.Filter("mctruth == 5").Count())

# Counts after each error code appearance
last_codes = ""
for code in counts_error_code.keys():
    filter = f"errorcode != {code} && !({last_codes})" if len(last_codes) > 0 else f"errorcode != {code}"
    
    counts_error_code[code] = df_Semileptonic.Filter(filter).Count()
    last_codes += f"|| errorcode == {code}" if len(last_codes) > 0 else f"errorcode == {code}"

ROOT.RDF.RunGraphs(stats_semileptonic["counts"])
ROOT.RDF.RunGraphs(list(counts_error_code.values()))

stats_semileptonic["efficiency"].append(stats_semileptonic["counts"][1].GetValue() / stats_semileptonic["counts"][0].GetValue())
stats_semileptonic["efficiency"].append(stats_semileptonic["counts"][2].GetValue() / stats_semileptonic["counts"][1].GetValue())

last_code = None
for code in counts_error_code.keys():
    efficiency_error_code[code] = counts_error_code[code].GetValue() / stats_semileptonic["counts"][0].GetValue()
    relative_efficiency_error_code[code] = efficiency_error_code[code] / efficiency_error_code[last_code] if last_code in efficiency_error_code else 0
    last_code = code

In [ ]:
print(f"Counts Semi Total: {stats_semileptonic['counts'][0].GetValue()} (100.00%)")
print(f"Counts Semi After Preselection: {stats_semileptonic['counts'][1].GetValue()} ({stats_semileptonic['efficiency'][0] * 100:.2f}%)")
print(f"Counts Semi After Cuts: {stats_semileptonic['counts'][2].GetValue()} ({stats_semileptonic['efficiency'][1] * 100:.2f}%)")

for code in counts_error_code.keys():
    print(f"Counts Semi After Error Code {code}: {counts_error_code[code].GetValue()} ({efficiency_error_code[code] * 100:.2f}%, Relative: {relative_efficiency_error_code[code] * 100:.2f}%)")

In [4]:
# Generate overlapped histograms for mctruth == 1
variables_1D = [#"Knerec_5", 
                #"KnerecSix_5", 
                "Kchrec_5"]
variables_1D_titles = [#"m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       #"m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]"]
range_X_1D = [#(100, -400, 200), 
              #(100, -400, 200), 
              (100, -200, 200)]

In [5]:
# 1D histograms for mctruth == 1
hist_1D = {}
modes = {"full": [], "zoom": []}

# Define histogram ranges for different modes
modes["full"] = [#(100, -400, 200), 
                 #(100, -400, 200), 
                 (100, -200, 200)]
modes["zoom"] = [#(100, -100, 100), 
                #(100, -100, 100), 
                (100, -5, 5)]

for i, var in enumerate(variables_1D):
  hist_1D[var] = {}
  title_str = f";{variables_1D_titles[i]};Counts"
  for mode in modes:
    hist_1D[var][mode] = df_Signal.Histo1D(
        (f"{var}_{mode}", title_str, *modes[mode][i]), var
    )

ROOT.RDF.RunGraphs(list(hist_1D[var][mode] for var in variables_1D for mode in modes))

1

In [6]:
ROOT.TGaxis.SetMaxDigits(4)
ROOT.gStyle.SetOptStat(0)

canvas = []
mode_list = list(modes.keys())  # ["full", "zoom"] (lub w kolejności, jakiej masz w słowniku)

for i, var in enumerate(variables_1D):
    canvas.append(ROOT.TCanvas(f"c_{i}", f"c_{i}", 800, 400))
    canvas[-1].Divide(len(mode_list), 1)
    
    # 1. ZAWSZE wykonujemy fit NA ZAKRESIE I HISTOGRAMIE "zoom"
    zoom_xmin = modes["zoom"][i][1]
    zoom_xmax = modes["zoom"][i][2]
    h_zoom = hist_1D[var]["zoom"]
    
    fit_func_zoom = ROOT.TF1(f"fit_zoom_{var}", ROOT.double_gaussian_cpp, zoom_xmin, zoom_xmax, 5)

    max_bin = h_zoom.GetMaximumBin()
    peak_center = h_zoom.GetBinCenter(max_bin)
    
    # Strzały początkowe bazujące na zoomie
    fit_func_zoom.SetParameter(0, h_zoom.GetEntries() * h_zoom.GetBinWidth(1))
    fit_func_zoom.SetParameter(1, peak_center)
    fit_func_zoom.SetParameter(2, h_zoom.GetRMS() * 0.5)
    fit_func_zoom.SetParameter(3, h_zoom.GetRMS() * 1.5)
    fit_func_zoom.SetParameter(4, 0.5)

    fit_func_zoom.SetParLimits(4, 0.0, 1.0)
    fit_func_zoom.SetParLimits(2, 0.0, h_zoom.GetRMS() * 1.0)  # sigma1 > 0
    fit_func_zoom.SetParLimits(3, 0.0, h_zoom.GetRMS() * 100.0)  # sigma2 > 0

    # Właściwy fit na zoomie (opcja "S" zwraca wynik, "R" ogranicza do zakresu TF1)
    fit_result = h_zoom.Fit(fit_func_zoom, "RSIL")

    # Wypisywanie wyników na konsolę
    print("=" * 60)
    print(f" WYNIKI FITU DLA ZMIENNEJ: {var}")
    print("=" * 60)
    if fit_result.IsValid():
        print(f" Status fitu       : Sukces (Valid)")
        print(f" Chi2 / NDF        : {fit_result.Chi2():.2f} / {fit_result.Ndf()} (p-value: {fit_result.Prob():.4g})")
        print("-" * 60)
        
        param_names = [
            "Amplituda (amp)", 
            "Średnia (mean)", 
            "Sigma 1 (sigma1)", 
            "Sigma 2 (sigma2)", 
            "Ułamek (fraction)"
        ]
        
        for p in range(5):
            val = fit_func_zoom.GetParameter(p)
            err = fit_func_zoom.GetParError(p)
            print(f" {param_names[p]:<20} = {val:12.4f} ± {err:.4f}")
    else:
        print(" OSTRZEŻENIE: Fit zakończył się niepowodzeniem (Invalid fit result)!")
    print("=" * 60 + "\n")

    # 2. Rysujemy wyniki fitu na obu padach (full oraz zoom), dostosowując zakres rysowania funkcji
    for j, mode in enumerate(mode_list):
        canvas[-1].cd(j + 1)

        curr_xmin = modes[mode][i][1]
        curr_xmax = modes[mode][i][2]
        h_curr = hist_1D[var][mode]

        h_curr.SetLineColor(ROOT.kBlack)
        h_curr.Draw("PE")

        if fit_result.IsValid():
            # Tworzymy funkcję do RYSOWANIA z zakresem dopasowanym do aktualnego trybu (full lub zoom)
            f_draw = ROOT.TF1(f"fit_draw_{var}_{mode}", ROOT.double_gaussian_cpp, curr_xmin, curr_xmax, 5)
            
            # Przepisujemy parametry uzyskane z fitu na zoomie
            for p in range(5):
                f_draw.SetParameter(p, fit_func_zoom.GetParameter(p))

            f_draw.SetLineColor(ROOT.kRed)
            f_draw.SetLineWidth(2)
            f_draw.Draw("SAME")

            # Tworzymy składowe (wąski i szeroki gauss) na pełny zakres bieżącego pada
            func_g1 = ROOT.TF1(f"g1_{var}_{mode}", ROOT.single_gauss_comp1, curr_xmin, curr_xmax, 4)
            func_g2 = ROOT.TF1(f"g2_{var}_{mode}", ROOT.single_gauss_comp2, curr_xmin, curr_xmax, 4)

            func_g1.SetParameters(f_draw.GetParameter(0), f_draw.GetParameter(1), 
                                 f_draw.GetParameter(2), f_draw.GetParameter(4))
            func_g2.SetParameters(f_draw.GetParameter(0), f_draw.GetParameter(1), 
                                 f_draw.GetParameter(3), f_draw.GetParameter(4))

            # Stylizacja składowych
            func_g1.SetLineColor(ROOT.kBlue)
            func_g1.SetLineStyle(2)  # przerywana
            func_g1.SetLineWidth(2)

            func_g2.SetLineColor(ROOT.kGreen + 2)
            func_g2.SetLineStyle(7)  # kropkowo-kreskowa
            func_g2.SetLineWidth(2)

            func_g1.Draw("SAME")
            func_g2.Draw("SAME")

    canvas[-1].Draw()

 WYNIKI FITU DLA ZMIENNEJ: Kchrec_5
 Status fitu       : Sukces (Valid)
 Chi2 / NDF        : 857.22 / 95 (p-value: 5.267e-123)
------------------------------------------------------------
 Amplituda (amp)      =   12500.6081 ± 37.0957
 Średnia (mean)       =      -0.0219 ± 0.0032
 Sigma 1 (sigma1)     =       0.8091 ± 0.0047
 Sigma 2 (sigma2)     =       2.3720 ± 0.0294
 Ułamek (fraction)    =       0.7231 ± 0.0050

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      428.612
Chi2                      =      857.223
NDf                       =           95
Edm                       =  3.44575e-06
NCalls                    =          133
p0                        =      12500.6   +/-   37.0957     
p1                        =   -0.0219138   +/-   0.00317404  
p2                        =     0.809089   +/-   0.00474204   	 (limited)
p3                        =      2.37201   +/-   0.0294024    	 (limited)
p4                        =     

In file included from input_line_167:1:
/opt/root/include/CPyCppyy/API.h:26:10: fatal error: 'Python.h' file not found
#include "Python.h"
         ^~~~~~~~~~
In file included from input_line_168:1:
/opt/root/include/CPyCppyy/API.h:26:10: fatal error: 'Python.h' file not found
#include "Python.h"
         ^~~~~~~~~~


In [15]:
variables_2D = [["Knerec_5", "KnerecSix_5"]]
variables_2D_titles = [["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]"]]
range_X_2D = [[(100, -400, 200), (100, -400, 200)]]

In [36]:
## 2D histograms for mctruth == 1
hist_2D = {}
modes = {}

# Definicja zakresów dla trybów: (nbinsX, xlow, xhigh, nbinsY, ylow, yhigh)
modes["full"] = (100, -400, 200, 100, -400, 200)
modes["zoom"] = (100, -100, 100, 100, -100, 100)

for varset in variables_2D:
    var_key = "_".join(varset)
    hist_2D[var_key] = {}
    title_str = f";{variables_2D_titles[0][0]};{variables_2D_titles[0][1]}"
    
    for mode, model_args in modes.items():
        hist_2D[var_key][mode] = df_Signal.Histo2D(
            (f"{var_key}_{mode}", title_str, *model_args),
            varset[0], varset[1]  # Przekazujemy obie zmienne dla 2D
        )

# Uruchomienie obliczeń dla wszystkich histogramów 2D
all_hists = [hist_2D[var_key][mode] for var_key in hist_2D for mode in modes]
ROOT.RDF.RunGraphs(all_hists)

1

In [37]:
ROOT.TGaxis.SetMaxDigits(6)

canva = ROOT.TCanvas("c2d", "2D Histograms", 400, 800)

canva.Divide(1, 2)

for j, mode in enumerate(modes):
  for i, var_key in enumerate(hist_2D):
    canva.cd(i * len(modes) + j +1)
    hist_2D[var_key][mode].Draw("COLZ")
  canva.Draw()
    

In [8]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = ["abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
            "Chi2SignalKinFit < 30.0"
            ]

In [7]:
signal_cut_stats = {}

total_signal = df_Signal.Count()

total_cut = ""
for cut in cut_list:
  if total_cut:
    total_cut += " && "
  total_cut += f"({cut})"

  signal_cut_stats[total_cut] = df_Signal.Filter(total_cut).Count()

ROOT.RDF.RunGraphs(list(signal_cut_stats.values()) + [total_signal])

print(f"Total signal count: {total_signal.GetValue()}")
for cut, count in signal_cut_stats.items():
  print(f"Cut: {cut}, Count: {count.GetValue()}.")


Total signal count: 164399
Cut: (abs(Kchrec_5) <= 3 * 0.8090), Count: 114440.
Cut: (abs(Kchrec_5) <= 3 * 0.8090) && (ellipse_cut_func(Knerec_5, KnerecSix_5)), Count: 109084.
Cut: (abs(Kchrec_5) <= 3 * 0.8090) && (ellipse_cut_func(Knerec_5, KnerecSix_5)) && ((rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0 && rho_omega > 1.5) || !(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0)), Count: 108054.
Cut: (abs(Kchrec_5) <= 3 * 0.8090) && (ellipse_cut_func(Knerec_5, KnerecSix_5)) && ((rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0 && rho_omega > 1.5) || !(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0)) && ((rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45 && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45)), Count: 107944.
Cut: (abs(Kchrec_5) <= 3 * 0